Installing VADER (Valence Aware Dictionary and sEntiment Reasoner), which is a rule-based sentiment analysis tool designed especially for social media text

In [1]:
!pip install vaderSentiment 

In [2]:
import pandas as pd
import numpy as np
import time

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

Loading the preprocessed data

In [3]:
df = pd.read_csv("sentimentalAnalysis_preprocessed.csv")

In [4]:
df = df.dropna(subset=['sentiment'])

In [5]:
df.shape

(1600000, 5)

In [6]:
df['sentiment'].value_counts()

sentiment
negative    800000
positive    800000
Name: count, dtype: int64

In [7]:
df.head()

,target,text,clean_text,partial_clean_text,sentiment
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",user url awww bummer shoulda got david carr th...,"USER - awww, that's a bummer. you shoulda got ...",negative
1,0,is upset that he can't update his Facebook by ...,upset cannot update facebook texting might cry...,is upset that he can't update his facebook by ...,negative
2,0,@Kenichan I dived many times for the ball. Man...,user dived many times ball managed save 50 res...,USER i dived many times for the ball. managed ...,negative
3,0,my whole body feels itchy and like its on fire,whole body feels itchy like fire,my whole body feels itchy and like its on fire,negative
4,0,"@nationwideclass no, it's not behaving at all....",user no not behaving mad cannot see,"USER no, it's not behaving at all. i'm mad. wh...",negative


In [10]:
y = df['sentiment'].str.lower().str.strip()
y = y.map({'negative': 0,
    'positive': 1})

print(y.value_counts())

sentiment
0    800000
1    800000
Name: count, dtype: int64


Function for dertermining sentiment based on polarity scores

In [11]:
analyzer = SentimentIntensityAnalyzer()

def predict_sentiment(text):

    score = analyzer.polarity_scores(str(text))['compound']

    if score >= 0.05:
        return 1

    elif score <= -0.05:
        return 0

    else:
        return 2

Function for prediction, accuracy, precision, f1, recall

In [12]:
def apply_prediction(data, column, version):

    start_time = time.time()

    predictions = data[column].fillna('').apply(predict_sentiment)

    end_time = time.time()

    execution_time = end_time - start_time

    valid = predictions != 2

    actual = y.loc[data.index][valid]
    predicted = predictions[valid]

    accuracy = accuracy_score(actual, predicted)
    precision = precision_score(actual, predicted, zero_division=0)
    recall = recall_score(actual, predicted, zero_division=0)
    f1 = f1_score(actual, predicted, zero_division=0)

    neutral_percentage = (predictions == 2).mean() * 100
    coverage = (predictions != 2).mean() * 100

    print(f"\n{version}")
    print("-" * 40)
    print("Execution Time:", round(execution_time, 4), "seconds")
    print("Accuracy:", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F1 Score:", round(f1, 4))
    print("Neutral Predictions:", round(neutral_percentage, 2), "%")

    return {
        'Version': version,
        'Execution Time': execution_time,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'Neutral Percentage': neutral_percentage
    }

In [13]:
results = []

results.append(
    apply_prediction(df, 'text', 'V1 - Original Text')
)

results.append(
    apply_prediction(df, 'clean_text', 'V2 - Clean Text')
)

results.append(
    apply_prediction(df, 'partial_clean_text', 'V3 - Partial Clean Text')
)


V1 - Original Text
----------------------------------------
Execution Time: 254.6522 seconds
Accuracy: 0.7181
Precision: 0.6649
Recall: 0.865
F1 Score: 0.7519
Neutral Predictions: 27.0 %

V2 - Clean Text
----------------------------------------
Execution Time: 179.4206 seconds
Accuracy: 0.7099
Precision: 0.6585
Recall: 0.8585
F1 Score: 0.7453
Neutral Predictions: 26.58 %

V3 - Partial Clean Text
----------------------------------------
Execution Time: 252.3275 seconds
Accuracy: 0.7183
Precision: 0.6652
Recall: 0.8649
F1 Score: 0.752
Neutral Predictions: 27.02 %


Comparision between different versions of preprocessing data

In [14]:
results_df = pd.DataFrame(results)
results_df

,Version,Execution Time,Accuracy,Precision,Recall,F1 Score,Neutral Percentage
0,V1 - Original Text,254.652226,0.718089,0.664906,0.865011,0.751872,27.002125
1,V2 - Clean Text,179.420553,0.709927,0.658529,0.858526,0.745344,26.577375
2,V3 - Partial Clean Text,252.327533,0.718297,0.665163,0.864888,0.751990,27.018687


For hadoop, I am taking text and target pairs as input file, save it to sentiment140_v1.tsv,
sentiment140_v2.tsv, sentiment140_v3.tsv

In [10]:
hadoop_df = pd.DataFrame({
    'target': df['sentiment'].map({
        'negative': 0,
        'positive': 1
    }),
    'text': df['text'].fillna('').astype(str)
})

hadoop_df['text'] = (
    hadoop_df['text']
    .str.replace(r'[\r\n\t]+', ' ', regex=True)
)

output_path = 'sentiment140_v1.tsv'

hadoop_df.to_csv(
    output_path,
    sep='\t',
    index=False,
    header=False
)

print("Saved:", output_path)
print("Rows:", len(hadoop_df))

Saved: sentiment140_v1.tsv
Rows: 1600000


In [12]:
hadoop_df = pd.DataFrame({
    'target': df['sentiment'].map({
        'negative': 0,
        'positive': 1
    }),
    'text': df['clean_text'].fillna('').astype(str)
})

hadoop_df['text'] = (
    hadoop_df['text']
    .str.replace(r'[\r\n\t]+', ' ', regex=True)
)

output_path = 'sentiment140_v2.tsv'

hadoop_df.to_csv(
    output_path,
    sep='\t',
    index=False,
    header=False
)

print("Saved:", output_path)
print("Rows:", len(hadoop_df))

Saved: sentiment140_v2.tsv
Rows: 1600000


In [13]:
hadoop_df = pd.DataFrame({
    'target': df['sentiment'].map({
        'negative': 0,
        'positive': 1
    }),
    'text': df['partial_clean_text'].fillna('').astype(str)
})

hadoop_df['text'] = (
    hadoop_df['text']
    .str.replace(r'[\r\n\t]+', ' ', regex=True)
)

output_path = 'sentiment140_v3.tsv'

hadoop_df.to_csv(
    output_path,
    sep='\t',
    index=False,
    header=False
)

print("Saved:", output_path)
print("Rows:", len(hadoop_df))

Saved: sentiment140_v3.tsv
Rows: 1600000
